# Kaggle House Prices — Reproducible Winning Baseline

Reconstructs the **original** CatBoost pipeline from the uploaded notebook: engineered property features, column-wise imputation/standardization/one-hot encoding inside each fold, CatBoost (1300 iterations, depth 5), five-fold OOF predictions, and the final `submission.csv`.

**Recorded original OOF log RMSE: 0.123416**. This is a past result, not a guarantee on a new runtime. The public leaderboard score is not known. The output file is a five-fold test-prediction average, just like the original notebook.

## 1 — Install packages

In [ ]:
%pip -q install -U kagglehub catboost scikit-learn pandas numpy matplotlib

## 2 — Get competition data

Accept the Kaggle competition rules first. Run `kagglehub.login()` only if your runtime needs authentication. No API credentials should be saved in this notebook.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
import kagglehub

SEED = 42
required = ['train.csv', 'test.csv', 'sample_submission.csv']
locations = [Path('/content'), Path('/content/data'), Path('/kaggle/input/house-prices-advanced-regression-techniques')]
data_dir = next((p for p in locations if all((p / f).is_file() for f in required)), None)
if data_dir is None:
    try:
        data_dir = Path(kagglehub.competition_download('house-prices-advanced-regression-techniques'))
    except Exception as exc:
        print('Dataset download needs Kaggle login. Authenticate and retry.', str(exc)[:160])
        kagglehub.login()
        data_dir = Path(kagglehub.competition_download('house-prices-advanced-regression-techniques'))
assert all((data_dir / f).is_file() for f in required), f'Missing files: {data_dir}'
train = pd.read_csv(data_dir / 'train.csv')
test = pd.read_csv(data_dir / 'test.csv')
sample = pd.read_csv(data_dir / 'sample_submission.csv')
assert train.shape == (1460, 81) and test.shape == (1459, 80)
assert sample.columns.tolist() == ['Id', 'SalePrice']
assert sample['Id'].equals(test['Id'])
print('Loaded:', data_dir, '| train', train.shape, '| test', test.shape)
display(train.head(3))

## 3 — Exact original feature engineering (no target-derived features)

In [ ]:
def features(df):
    x = df.drop(columns=['Id', 'SalePrice'], errors='ignore').copy()
    for c in ['MSSubClass', 'MoSold']:
        if c in x:
            x[c] = x[c].astype('string').fillna('Missing').astype(str)
    def col(name):
        return pd.to_numeric(x[name], errors='coerce').fillna(0) if name in x else pd.Series(0, index=x.index)
    x['TotalFinishedSF'] = col('GrLivArea') + col('TotalBsmtSF')
    x['TotalBathrooms'] = col('FullBath') + 0.5 * col('HalfBath') + col('BsmtFullBath') + 0.5 * col('BsmtHalfBath')
    x['TotalPorchSF'] = sum((col(c) for c in ['OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'WoodDeckSF']), start=pd.Series(0, index=x.index))
    x['HouseAgeAtSale'] = (col('YrSold') - col('YearBuilt')).clip(lower=0)
    x['YearsSinceRemodel'] = (col('YrSold') - col('YearRemodAdd')).clip(lower=0)
    return x

X = features(train)
X_test = features(test)
y = np.log1p(train['SalePrice'].to_numpy())
assert list(X.columns) == list(X_test.columns)
cat_cols = X.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]
print('Feature matrix:', X.shape, '| numeric:', len(num_cols), '| categorical:', len(cat_cols))

## 4 — Original preprocessing and CatBoost configuration

In [ ]:
from sklearn.model_selection import KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.base import clone
from sklearn.metrics import mean_squared_error
from catboost import CatBoostRegressor

preprocessor = ColumnTransformer([
    ('num', Pipeline([('fill', SimpleImputer(strategy='median', add_indicator=True)), ('scale', StandardScaler())]), num_cols),
    ('cat', Pipeline([('fill', SimpleImputer(strategy='constant', fill_value='Missing')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), cat_cols),
])
catboost = CatBoostRegressor(
    iterations=1300, depth=5, learning_rate=0.03,
    l2_leaf_reg=5, loss_function='RMSE', verbose=False,
    thread_count=2, random_seed=SEED, allow_writing_files=False,
)
folds = list(KFold(n_splits=5, shuffle=True, random_state=SEED).split(X))

## 5 — Train five folds; record OOF and test predictions

In [ ]:
oof_catboost = np.zeros(len(X))
test_fold_preds = []
fold_scores = []
for fold, (tr, va) in enumerate(folds, 1):
    pipe = Pipeline([('prep', clone(preprocessor)), ('model', clone(catboost))])
    pipe.fit(X.iloc[tr], y[tr])
    oof_catboost[va] = pipe.predict(X.iloc[va])
    test_fold_preds.append(pipe.predict(X_test))
    score = np.sqrt(mean_squared_error(y[va], oof_catboost[va]))
    fold_scores.append(score)
    print(f'Fold {fold}: {score:.6f}')

catboost_score = np.sqrt(mean_squared_error(y, oof_catboost))
final_log_pred = np.mean(test_fold_preds, axis=0)
print(f'\nOriginal-recipe CatBoost 5-fold OOF log RMSE: {catboost_score:.6f}')
print('Previously recorded from your notebook: 0.123416')

## 6 — Validate and export Kaggle submission

Test predictions are the average of five fold-trained models, inverted from `log1p` via `expm1`. Kaggle expects the exact columns `Id,SalePrice`. No test target values are available locally.

In [ ]:
from pathlib import Path
output_dir = Path('/content') if Path('/content').exists() else Path.cwd()
submission_path = output_dir / 'submission.csv'
submission = sample[['Id']].copy()
submission['SalePrice'] = np.maximum(np.expm1(final_log_pred), 1.0)
assert submission.shape == (1459, 2)
assert submission['Id'].equals(test['Id'])
assert submission['Id'].is_unique
assert submission['SalePrice'].notna().all()
assert np.isfinite(submission['SalePrice']).all()
assert (submission['SalePrice'] > 0).all()
submission.to_csv(submission_path, index=False)
print('Saved:', submission_path)
display(submission.head())
try:
    from google.colab import files
    files.download(str(submission_path))
except ImportError:
    pass

## 7 — Optional diagnostics (does not affect submission)

Use residuals to understand difficult houses. Do not remove validation examples to report an artificially lower CV metric. The recorded 0.123416 score and the new execution score are both cross-validation estimates, **not Kaggle leaderboard results**.

In [ ]:
import matplotlib.pyplot as plt
errors = y - oof_catboost
print('OOF RMSE:', round(catboost_score, 6))
print('Squared-error share from IDs 524 and 1299:',
      round(100 * np.sum(errors[train['Id'].isin([524, 1299]).to_numpy()] ** 2) / np.sum(errors ** 2), 2), '%')
fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(y, errors, alpha=0.45, s=16)
ax.axhline(0, linestyle='--', color='gray')
ax.set(xlabel='True log1p(SalePrice)', ylabel='OOF residual', title='CatBoost OOF residuals')
plt.tight_layout()
plt.show()